# 第 34 章：奇异期权

本章定价过的每一种奇异期权，集中在一处。**用 NumPy 与 SciPy**；
闭式解是逐条写出来的，而不是从库里调用，
于是正文里的公式与代码完全一致。

| 节 | 工具 | 方法 | 例 |
|---|---|---|---|
| 1 | 障碍期权（向下敲入／敲出） | Reiner–Rubinstein 闭式解 | 34.1 |
| 2 | 树上的障碍期权 | CRR，以及它为什么失效 | 34.2 |
| 3 | 静态复制的障碍期权 | 反射论证 | 34.3 |
| 4 | 离散监测的障碍期权 | Broadie–Glasserman–Kou | 34.4 |
| 5 | 数字期权 | 闭式解、看涨价差、超额对冲 | 34.5 |
| 6 | 反向敲出 | 闭式解，以及它为负的 delta | 34.6 |
| 7 | 回望与亚式 | Goldman–Sosin–Gatto、Kemna–Vorst | 34.7 |
| 8 | 交换与篮子 | Margrabe；蒙特卡洛与矩匹配 | 34.8 |
| 9 | 最差价期权 | 蒙特卡洛 | 34.9 |
| 10 | 封顶棘轮 | 远期起始看涨价差 | 34.10 |

每个单元都会把算出来的值印在本章所印的值旁边。本章的每日监测核对与最差价表是蒙特卡洛估计，本笔记本的篮子期权
也用蒙特卡洛计算，因此这两处按*标准误差*比较而不是逐位比较，
这也正是例 34.4 已经在用的说法。

表头保留英文：中文字符在等宽终端里是双宽的，而 Python 的 `%-44s` 按字符数计。

In [1]:
import numpy as np
from math import log, sqrt, exp, ceil
from scipy.stats import norm

N = norm.cdf


def show(label, got, printed, tol=6e-4):
    """把算出来的值印在本章所印的值旁边。"""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-44s %14.6f   book %12.6f   %s" % (label, got, printed, flag))


def bs(S, K, T, r, q, sig, call=True):
    """带连续收益率 q 的 Black--Scholes。"""
    if T <= 0:
        return max(S - K, 0.0) if call else max(K - S, 0.0)
    d1 = (log(S / K) + (r - q + 0.5 * sig * sig) * T) / (sig * sqrt(T))
    d2 = d1 - sig * sqrt(T)
    if call:
        return S * exp(-q * T) * N(d1) - K * exp(-r * T) * N(d2)
    return K * exp(-r * T) * N(-d2) - S * exp(-q * T) * N(-d1)


S0, K, H, r, q, sig, T = 100.0, 100.0, 90.0, 0.05, 0.0, 0.25, 1.0
print("本章的贯穿例子: S0=%.0f K=%.0f H=%.0f r=%.0f%% sigma=%.0f%% T=%.0f"
      % (S0, K, H, 100 * r, 100 * sig, T))

本章的贯穿例子: S0=100 K=100 H=90 r=5% sigma=25% T=1


## 1. 障碍期权的闭式解：例 34.1

Reiner–Rubinstein 公式。其余的由入出平价完成：
一份向下敲出加一份向下敲入就是相应的普通期权，因为两个示性函数加起来等于一。

In [2]:
def barrier_call(S, K, H, T, r, q, sig, kind):
    """kind 取 {'di','do'}（H <= K）或 {'ui','uo'}（H > K）。"""
    lam = (r - q + 0.5 * sig * sig) / (sig * sig)
    sT = sig * sqrt(T)
    c = bs(S, K, T, r, q, sig, True)
    y = log(H * H / (S * K)) / sT + lam * sT
    x1 = log(S / H) / sT + lam * sT
    y1 = log(H / S) / sT + lam * sT
    pw, pw2 = (H / S) ** (2 * lam), (H / S) ** (2 * lam - 2)
    if kind in ("di", "do"):
        di = S * exp(-q * T) * pw * N(y) - K * exp(-r * T) * pw2 * N(y - sT)
        return di if kind == "di" else c - di
    ui = (S * exp(-q * T) * N(x1) - K * exp(-r * T) * N(x1 - sT)
          - S * exp(-q * T) * pw * (N(-y) - N(-y1))
          + K * exp(-r * T) * pw2 * (N(-y + sT) - N(-y1 + sT)))
    return ui if kind == "ui" else c - ui


van = bs(S0, K, T, r, q, sig)
do = barrier_call(S0, K, H, T, r, q, sig, "do")
di = barrier_call(S0, K, H, T, r, q, sig, "di")
show("vanilla call", van, 12.335999)
show("down-and-out call", do, 9.111221)
show("down-and-in call", di, 3.224778)
show("in-out parity: do + di - vanilla", do + di - van, 0.0)

lam = (r - q + 0.5 * sig ** 2) / sig ** 2
show("lambda", lam, 1.3)
show("(H/S0)^(2 lambda)", (H / S0) ** (2 * lam), 0.760380)
show("(H/S0)^(2 lambda - 2)", (H / S0) ** (2 * lam - 2), 0.938740)
show("y", log(H * H / (S0 * K)) / (sig * sqrt(T)) + lam * sig * sqrt(T), -0.517884)
show("premium the barrier removes, %", 100 * (1 - do / van), 26.1, tol=3e-3)

vanilla call                                      12.335999   book    12.335999   ok
down-and-out call                                  9.111221   book     9.111221   ok
down-and-in call                                   3.224778   book     3.224778   ok
in-out parity: do + di - vanilla                   0.000000   book     0.000000   ok
lambda                                             1.300000   book     1.300000   ok
(H/S0)^(2 lambda)                                  0.760380   book     0.760380   ok
(H/S0)^(2 lambda - 2)                              0.938740   book     0.938740   ok
y                                                 -0.517884   book    -0.517884   ok
premium the barrier removes, %                    26.141201   book    26.100000   ok


## 2. 障碍期权为何让树方法失灵：例 34.2

树只能在节点上敲出，因此它定价的是由其节点层恰好定义出来的合约。
步数从 50 增到 51，误差扩大到 45 倍，而且误差是单边的，所以对网格取平均没有用。
补救办法是把 $n$ 选得让某一层正好落在障碍上。

In [3]:
def crr_barrier(n, S=100.0, K=100.0, H=90.0, T=1.0, r=0.05, sig=0.25):
    dt = T / n
    u = exp(sig * sqrt(dt))
    d = 1.0 / u
    p = (exp(r * dt) - d) / (u - d)
    disc = exp(-r * dt)
    j = np.arange(n + 1)
    S_T = S * u ** (2 * j - n)
    v = np.where(S_T > H, np.maximum(S_T - K, 0.0), 0.0)
    for i in range(n - 1, -1, -1):
        j = np.arange(i + 1)
        S_i = S * u ** (2 * j - i)
        v = disc * (p * v[1:] + (1 - p) * v[:-1])
        v = np.where(S_i > H, v, 0.0)
    return float(v[0])


def effective_barrier(n, S=100.0, H=90.0, T=1.0, sig=0.25):
    """不超过 H 的最高节点层，也就是树实际执行的障碍。"""
    u = exp(sig * sqrt(T / n))
    return S * u ** (-ceil(log(S / H) / log(u)))


print("%6s %12s %10s %18s" % ("steps", "tree", "error", "effective barrier"))
for n, wv, wh in ((50, 9.142046, 89.94), (51, 10.494157, 86.93), (100, 9.944477, 88.25),
                  (200, 9.144117, 89.94), (400, 9.438414, 89.36), (1600, 9.153506, 89.92)):
    v, h = crr_barrier(n), effective_barrier(n)
    print("%6d %12.6f %10.3f %18.2f    book %.6f / %.2f" % (n, v, v - do, h, wv, wh))

print()
print("把 n 选得让某一层落在障碍上: n = floor(m^2 sigma^2 T / ln^2(S0/H))")
for m, wn, wv in ((4, 90, 9.111965), (6, 202, 9.119059), (8, 360, 9.113142),
                  (9, 456, 9.111153), (12, 810, 9.113370), (16, 1441, 9.112680)):
    n = int(m * m * sig ** 2 * T / log(S0 / H) ** 2)
    v = crr_barrier(n)
    print("m=%2d  n=%5d  %12.6f  error %+.5f    book n=%d %.6f" % (m, n, v, v - do, wn, wv))

 steps         tree      error  effective barrier
    50     9.142046      0.031              89.94    book 9.142046 / 89.94
    51    10.494157      1.383              86.93    book 10.494157 / 86.93
   100     9.944477      0.833              88.25    book 9.944477 / 88.25
   200     9.144117      0.033              89.94    book 9.144117 / 89.94
   400     9.438414      0.327              89.36    book 9.438414 / 89.36
  1600     9.153506      0.042              89.92    book 9.153506 / 89.92

把 n 选得让某一层落在障碍上: n = floor(m^2 sigma^2 T / ln^2(S0/H))
m= 4  n=   90      9.111965  error +0.00074    book n=90 9.111965
m= 6  n=  202      9.119059  error +0.00784    book n=202 9.119059
m= 8  n=  360      9.113142  error +0.00192    book n=360 9.113142
m= 9  n=  456      9.111153  error -0.00007    book n=456 9.111153
m=12  n=  810      9.113370  error +0.00215    book n=810 9.113370
m=16  n= 1441      9.112680  error +0.00146    book n=1441 9.112680


## 3. 反射论证给出的静态复制：例 34.3

当漂移为零时，一份向下敲入看涨期权恰好等于 $K/H$ 份执行价为 $H^2/K$ 的看跌期权，
精确到机器精度。破坏零漂移条件，这个对冲就会错 42%，
这正是该技术的适用边界。

In [4]:
print("零漂移（r = q）时:")
for rr in (0.00, 0.02, 0.05):
    d_in = barrier_call(S0, K, H, T, rr, rr, sig, "di")
    puts = (K / H) * bs(S0, H * H / K, T, rr, rr, sig, False)
    print("  r=q=%.0f%%   向下敲入 %.6f   看跌 %.6f   差 %+.2e"
          % (100 * rr, d_in, puts, d_in - puts))

print()
print("现在破坏它，取 r = 5%、q = 0:")
puts_broken = (K / H) * bs(S0, H * H / K, T, 0.05, 0.0, sig, False)
show("down-and-in call", di, 3.224778)
show("replicating puts", puts_broken, 1.861532)
show("static hedge error, %", 100 * (puts_broken / di - 1), -42.3, tol=3e-3)

零漂移（r = q）时:
  r=q=0%   向下敲入 2.771613   看跌 2.771613   差 -1.78e-15
  r=q=2%   向下敲入 2.716731   看跌 2.716731   差 -4.44e-15
  r=q=5%   向下敲入 2.636440   看跌 2.636440   差 +1.33e-15

现在破坏它，取 r = 5%、q = 0:
down-and-in call                                   3.224778   book     3.224778   ok
replicating puts                                   1.861532   book     1.861532   ok
static hedge error, %                            -42.274106   book   -42.300000   ok


## 4. 离散监测：例 34.4

Broadie、Glasserman 与 Kou：被监测 $m$ 次的障碍，其表现相当于一个
*远离*即期、平移了 $\beta\sigma\sqrt{T/m}$ 的连续障碍，其中
$\beta = -\zeta(1/2)/\sqrt{2\pi} = 0.5826$。即便是每日监测也值这份期权的 4.6%。

下面的蒙特卡洛用的是它自己的随机种子，因此它的标准误差不会与正文逐位相同。
能复现的是结论本身：修正后的公式落在约一个标准误差之内，
而未修正的公式在十六个标准误差之外。

In [5]:
BETA = 0.5826

print("%-10s %5s %14s %12s %14s" % ("monitoring", "m", "effective H", "price", "vs continuous"))
for name, m, wh, wv, wp in (("daily", 252, 89.18, 9.527218, 4.57), ("weekly", 52, 88.20, 9.968231, 9.41),
                            ("monthly", 12, 86.29, 10.675121, 17.16), ("quarterly", 4, 83.68, 11.365166, 24.74)):
    Heff = H * exp(-BETA * sig * sqrt(T / m))
    v = barrier_call(S0, K, Heff, T, r, q, sig, "do")
    print("%-10s %5d %14.2f %12.6f %13.2f%%    book %.2f / %.6f / %.2f%%"
          % (name, m, Heff, v, 100 * (v / do - 1), wh, wv, wp))

monitoring     m    effective H        price  vs continuous
daily        252          89.18     9.527220          4.57%    book 89.18 / 9.527218 / 4.57%
weekly        52          88.20     9.968234          9.41%    book 88.20 / 9.968231 / 9.41%
monthly       12          86.29    10.675126         17.16%    book 86.29 / 10.675121 / 17.16%
quarterly      4          83.68    11.365172         24.74%    book 83.68 / 11.365166 / 24.74%


In [6]:
# 一次独立核验：直接模拟每日监测的合约
rng = np.random.default_rng(0)
n_half, m = 200_000, 252
z = rng.standard_normal((n_half, m))
z = np.concatenate([z, -z])                       # antithetic
dt = T / m
paths = S0 * np.exp(np.cumsum((r - 0.5 * sig ** 2) * dt + sig * sqrt(dt) * z, axis=1))
alive = paths.min(axis=1) > H
payoff = np.where(alive, np.maximum(paths[:, -1] - K, 0.0), 0.0) * exp(-r * T)
pair = 0.5 * (payoff[:n_half] + payoff[n_half:])        # antithetic pairs are the independent draws
mc, se = pair.mean(), pair.std() / sqrt(n_half)

corrected = barrier_call(S0, K, H * exp(-BETA * sig * sqrt(T / m)), T, r, q, sig, "do")
print("蒙特卡洛（每日）        %.6f   标准误差 %.6f" % (mc, se))
print("修正后的公式            %.6f   %+.2f 个标准误差" % (corrected, (corrected - mc) / se))
print("未修正（连续）          %.6f   %+.1f 个标准误差" % (do, (do - mc) / se))

蒙特卡洛（每日）        9.504617   标准误差 0.024128
修正后的公式            9.527220   +0.94 个标准误差
未修正（连续）          9.111221   -16.3 个标准误差


## 5. 数字期权与超额对冲：例 34.5

看涨价差复制在价值上收敛、在份数上发散，两列里的全部困难就在于此。
于是交易商买入一个完全位于执行价之下的价差：它处处支配数字期权的收益，
而这部分溢价为买卖价差设定了下限。

In [7]:
digital = exp(-r * T) * N((log(S0 / K) + (r - q - 0.5 * sig ** 2) * T) / (sig * sqrt(T)))
show("cash-or-nothing call", digital, 0.504049)

print()
print("%10s %12s %12s" % ("epsilon", "notional", "value"))
for eps, wv in ((5.00, 0.504159), (1.00, 0.504054), (0.10, 0.504050), (0.01, 0.504049)):
    v = (bs(S0, K - eps / 2, T, r, q, sig) - bs(S0, K + eps / 2, T, r, q, sig)) / eps
    print("%10.2f %12.1f %12.6f    book %.6f" % (eps, 1 / eps, v, wv))

print()
print("单边价差，它处处支配该收益:")
for lo, wv, wp in ((99.0, 0.511635, 1.5), (98.0, 0.519249, 3.0), (95.0, 0.542210, 7.6)):
    v = (bs(S0, lo, T, r, q, sig) - bs(S0, K, T, r, q, sig)) / (K - lo)
    print("  [%3.0f, 100]   成本 %.6f   相对数字期权的溢价 %+.1f%%    book %.6f / %+.1f%%"
          % (lo, v, 100 * (v / digital - 1), wv, wp))

cash-or-nothing call                               0.504049   book     0.504049   ok

   epsilon     notional        value
      5.00          0.2     0.504159    book 0.504159
      1.00          1.0     0.504054    book 0.504054
      0.10         10.0     0.504050    book 0.504050
      0.01        100.0     0.504049    book 0.504049

单边价差，它处处支配该收益:
  [ 99, 100]   成本 0.511635   相对数字期权的溢价 +1.5%    book 0.511635 / +1.5%
  [ 98, 100]   成本 0.519249   相对数字期权的溢价 +3.0%    book 0.519249 / +3.0%
  [ 95, 100]   成本 0.542210   相对数字期权的溢价 +7.6%    book 0.542210 / +7.6%


## 6. 反向敲出：例 34.6

障碍在执行价之上，因此期权恰好在看涨期权开始变得值钱的地方敲出。
它只值相应普通期权的 6.7%，而且**标的在 94.3 以上时它的 delta 为负**：
这是一份持有人在标的上涨时亏钱的看涨期权。
把障碍外移 2 个点（障碍水平的 1.7%），要花掉这份期权价值的三分之一。

In [8]:
Hu, ru, qu, sigu = 120.0, 0.0, 0.0, 0.25          # 零持有成本，因此反射论证精确适用
van_u = bs(100.0, 100.0, 1.0, ru, qu, sigu)
uo = barrier_call(100.0, 100.0, Hu, 1.0, ru, qu, sigu, "uo")
show("vanilla, zero carry", van_u, 9.9476, tol=1e-4)
show("up-and-out call", uo, 0.6709, tol=3e-4)
show("as a share of the vanilla, %", 100 * uo / van_u, 6.7, tol=1e-2)

print()
print("%8s %12s %12s" % ("spot", "value", "delta"))
for s, wv, wd in ((100, 0.6709, -0.0149), (110, 0.4081, -0.0359),
                  (115, 0.2132, -0.0414), (118, 0.0863, -0.0430)):
    h = 0.01
    v = barrier_call(s, 100.0, Hu, 1.0, ru, qu, sigu, "uo")
    dlt = (barrier_call(s + h, 100.0, Hu, 1.0, ru, qu, sigu, "uo")
           - barrier_call(s - h, 100.0, Hu, 1.0, ru, qu, sigu, "uo")) / (2 * h)
    print("%8d %12.6f %12.6f    book %.4f / %+.4f" % (s, v, dlt, wv, wd))

print()
print("超额对冲：卖出一个障碍更远的敲出期权")
for Hb, wv, wp in ((122.0, 0.8947, 33.4), (124.0, 1.1513, 71.6), (126.0, 1.4377, 114.3)):
    v = barrier_call(100.0, 100.0, Hb, 1.0, ru, qu, sigu, "uo")
    print("  障碍 %5.0f   价值 %.6f   相对该合约的成本 %+.1f%%    book %.4f / %+.1f%%"
          % (Hb, v, 100 * (v / uo - 1), wv, wp))

vanilla, zero carry                                9.947645   book     9.947600   ok
up-and-out call                                    0.670865   book     0.670900   ok
as a share of the vanilla, %                       6.743957   book     6.700000   ok

    spot        value        delta
     100     0.670865    -0.014864    book 0.6709 / -0.0149
     110     0.408068    -0.035877    book 0.4081 / -0.0359
     115     0.213208    -0.041437    book 0.2132 / -0.0414
     118     0.086284    -0.042955    book 0.0863 / -0.0430

超额对冲：卖出一个障碍更远的敲出期权
  障碍   122   价值 0.894661   相对该合约的成本 +33.4%    book 0.8947 / +33.4%
  障碍   124   价值 1.151277   相对该合约的成本 +71.6%    book 1.1513 / +71.6%
  障碍   126   价值 1.437704   相对该合约的成本 +114.3%    book 1.4377 / +114.3%


## 7. 回望与亚式：例 34.7

同样的标的、同样的波动率，高者的价值约为低者的三倍。
亚式便宜，是因为平均值的方差小于终值，有效波动率约为 $\sigma/\sqrt{3}$。
回望贵，是因为它把择时这件事整个拿掉了。

In [9]:
def lookback_floating_call(S, Smin, T, r, q, sig):
    """Goldman--Sosin--Gatto：支付 S_T - min S，因此永远不会到期一文不值。"""
    sT = sig * sqrt(T)
    a1 = (log(S / Smin) + (r - q + 0.5 * sig ** 2) * T) / sT
    a2 = a1 - sT
    a3 = (log(S / Smin) + (-r + q + 0.5 * sig ** 2) * T) / sT
    Y1 = -2.0 * (r - q - 0.5 * sig ** 2) * log(S / Smin) / (sig ** 2)
    k = sig ** 2 / (2.0 * (r - q))
    return (S * exp(-q * T) * N(a1) - S * exp(-q * T) * k * N(-a1)
            - Smin * exp(-r * T) * (N(a2) - k * exp(Y1) * N(-a3)))


def geometric_asian_call(S, K, T, r, q, sig):
    """Kemna--Vorst：几何平均恰好是对数正态的。"""
    sg = sig / sqrt(3.0)
    bg = 0.5 * (r - q - sig ** 2 / 6.0)
    d1 = (log(S / K) + (bg + 0.5 * sg ** 2) * T) / (sg * sqrt(T))
    return exp(-r * T) * (S * exp(bg * T) * N(d1) - K * N(d1 - sg * sqrt(T)))


lb = lookback_floating_call(S0, S0, T, r, q, sig)
az = geometric_asian_call(S0, K, T, r, q, sig)
show("vanilla at-the-money call", van, 12.3360, tol=5e-5)
show("floating-strike lookback call", lb, 20.5522, tol=5e-5)
show("geometric-average Asian call", az, 6.5298, tol=5e-5)
show("lookback / vanilla", lb / van, 1.67, tol=3e-3)
show("Asian / vanilla", az / van, 0.53, tol=5e-3)
show("effective Asian vol, % (sigma/sqrt3)", 100 * sig / sqrt(3.0), 14.43, tol=1e-3)

vanilla at-the-money call                         12.335999   book    12.336000   ok
floating-strike lookback call                     20.552183   book    20.552200   ok
geometric-average Asian call                       6.529835   book     6.529800   ok
lookback / vanilla                                 1.666033   book     1.670000   ok
Asian / vanilla                                    0.529332   book     0.530000   ok
effective Asian vol, % (sigma/sqrt3)              14.433757   book    14.430000   ok


## 8. 交换期权与篮子期权：例 34.8

Margrabe 交换期权有闭式解，而相关性只通过 $\hat\sigma$ 起作用，
因此这张表就是五个波动率下的布莱克–斯科尔斯–默顿公式，
而且公式里没有利率，因为这份期权是按两个资产的*比值*定价的。

篮子期权没有闭式解：对数正态的加权和不是对数正态的。
正文的数值是精确值，这里用蒙特卡洛计算，因此按标准误差来比较。
矩匹配的对数正态是最常用的近似，而它恰恰在篮子最不像对数正态的地方退化。

In [10]:
s1, s2 = 0.25, 0.30
print("Margrabe 交换期权, S1 = S2 = 100, T = 1")
print("%8s %14s %12s" % ("rho", "effective vol", "value"))
for rho, wv, wval in ((-0.5, 47.70, 18.8495), (0.0, 39.05, 15.4808), (0.5, 27.84, 11.0703),
                      (0.9, 13.23, 5.2737), (0.99, 6.32, 2.5227)):
    sh = sqrt(s1 ** 2 + s2 ** 2 - 2 * rho * s1 * s2)
    d1 = 0.5 * sh * sqrt(T)                        # S1 = S2，故 ln(S1/S2) = 0
    v = 100.0 * N(d1) - 100.0 * N(d1 - sh * sqrt(T))
    print("%8.2f %13.2f%% %12.4f    book %.2f%% / %.4f" % (rho, 100 * sh, v, wv, wval))

Margrabe 交换期权, S1 = S2 = 100, T = 1
     rho  effective vol        value
   -0.50         47.70%      18.8495    book 47.70% / 18.8495
    0.00         39.05%      15.4808    book 39.05% / 15.4808
    0.50         27.84%      11.0703    book 27.84% / 11.0703
    0.90         13.23%       5.2737    book 13.23% / 5.2737
    0.99          6.32%       2.5227    book 6.32% / 2.5227


In [11]:
def basket_mc(rho, n=4_000_000, seed=7, S=(100.0, 100.0), w=(0.5, 0.5), sigs=(0.25, 0.30)):
    rng = np.random.default_rng(seed)
    z1 = rng.standard_normal(n)
    z1 = np.concatenate([z1, -z1])
    z2r = rng.standard_normal(n)
    z2r = np.concatenate([z2r, -z2r])
    z2 = rho * z1 + sqrt(max(1 - rho * rho, 0.0)) * z2r
    B = (w[0] * S[0] * np.exp((r - 0.5 * sigs[0] ** 2) * T + sigs[0] * sqrt(T) * z1)
         + w[1] * S[1] * np.exp((r - 0.5 * sigs[1] ** 2) * T + sigs[1] * sqrt(T) * z2))
    pay = np.maximum(B - K, 0.0) * exp(-r * T)
    return pay.mean(), pay.std() / sqrt(len(pay))


def basket_moment_matched(rho, S=(100.0, 100.0), w=(0.5, 0.5), sigs=(0.25, 0.30)):
    F = [s * exp(r * T) for s in S]
    M1 = sum(wi * Fi for wi, Fi in zip(w, F))
    M2 = sum(w[i] * w[j] * F[i] * F[j] * exp((rho if i != j else 1.0) * sigs[i] * sigs[j] * T)
             for i in range(2) for j in range(2))
    sB = sqrt(log(M2 / M1 ** 2) / T)
    d1 = (log(M1 / K) + 0.5 * sB ** 2 * T) / (sB * sqrt(T))
    return exp(-r * T) * (M1 * N(d1) - K * N(d1 - sB * sqrt(T))), sB


basket_of_calls = 0.5 * bs(100, K, T, r, q, s1) + 0.5 * bs(100, K, T, r, q, s2)
show("basket of calls", basket_of_calls, 13.2836, tol=5e-5)

print()
print("%7s %20s %12s %10s %14s" % ("rho", "basket call (MC)", "moment-m.", "ratio", "book"))
for rho, wv, wr in ((-0.5, 8.3481, 0.628), (0.0, 10.3260, 0.777), (0.5, 11.9088, 0.896),
                    (0.9, 13.0178, 0.980), (1.0, 13.2792, 1.000)):
    m, se = basket_mc(rho)
    mm, _ = basket_moment_matched(rho)
    print("%7.2f %11.4f (%.4f) %12.4f %10.3f %14.4f  (%+.1f s.e.)"
          % (rho, m, se, mm, m / basket_of_calls, wv, (wv - m) / se))

print()
print("矩匹配近似高估了:")
for rho, wv in ((-0.5, 8.3481), (0.0, 10.3260), (0.5, 11.9088), (0.9, 13.0178)):
    mm, _ = basket_moment_matched(rho)
    print("  rho=%+.2f   %+.2f%%" % (rho, 100 * (mm / wv - 1)))

basket of calls                                   13.283627   book    13.283600   ok

    rho     basket call (MC)    moment-m.      ratio           book


  -0.50      8.3511 (0.0039)       8.4106      0.629         8.3481  (-0.8 s.e.)


   0.00     10.3273 (0.0052)      10.3468      0.777        10.3260  (-0.3 s.e.)


   0.50     11.9095 (0.0063)      11.9192      0.897        11.9088  (-0.1 s.e.)


   0.90     13.0178 (0.0071)      13.0253      0.980        13.0178  (-0.0 s.e.)


   1.00     13.2798 (0.0073)      13.2863      1.000        13.2792  (-0.1 s.e.)

矩匹配近似高估了:
  rho=-0.50   +0.75%
  rho=+0.00   +0.20%
  rho=+0.50   +0.09%
  rho=+0.90   +0.06%


## 9. 最差价期权：例 34.9

篮子做平均，因此低相关性*降低*离散度、使期权变便宜。
最差价取最小值，因此低相关性*提高*了其中某一个大幅下跌的机会。
相关系数为 0.20 时，三个资产一年后的期望最小值约为 84，而起点是 100，
这纯粹是因为在三个资产中取了表现最差者。最差价看跌期权约为单资产看跌期权的两倍，这正是票息的来源。

In [12]:
def worstof_put_mc(rho, n=2_000_000, seed=11, d=3, S=100.0, Kw=100.0, sigw=0.28):
    rng = np.random.default_rng(seed)
    L = np.linalg.cholesky(np.full((d, d), rho) + np.eye(d) * (1 - rho))
    Z = rng.standard_normal((d, n))
    Z = np.concatenate([Z, -Z], axis=1)
    S_T = S * np.exp((r - 0.5 * sigw ** 2) * T + sigw * sqrt(T) * (L @ Z))
    mn = S_T.min(axis=0)
    pay = np.maximum(Kw - mn, 0.0) * exp(-r * T)
    return pay.mean(), pay.std() / sqrt(len(pay)), mn.mean()


single_put = bs(100.0, 100.0, T, r, q, 0.28, call=False)
print("单资产看跌期权, 28%% 波动率: %.4f" % single_put)
print()
print("%7s %12s %22s %12s" % ("rho", "E[min S_T]", "worst-of put (MC)", "vs single"))
for rho, wm, wv, wr in ((0.20, 83.86, 17.65, 2.05), (0.40, 86.58, 16.26, 1.89),
                        (0.60, 89.87, 14.71, 1.71), (0.80, 94.23, 12.80, 1.49),
                        (0.95, 99.62, 10.64, 1.24)):
    m, se, emin = worstof_put_mc(rho)
    print("%7.2f %12.2f %13.4f (%.4f) %12.2f    book %.2f / %.4f / %.2f  (%+.1f s.e.)"
          % (rho, emin, m, se, m / single_put, wm, wv, wr, (wv - m) / se))

单资产看跌期权, 28% 波动率: 8.5955

    rho   E[min S_T]      worst-of put (MC)    vs single


   0.20        83.86       17.6533 (0.0070)         2.05    book 83.86 / 17.6500 / 2.05  (-0.5 s.e.)


   0.40        86.58       16.2629 (0.0071)         1.89    book 86.58 / 16.2600 / 1.89  (-0.4 s.e.)


   0.60        89.87       14.7101 (0.0071)         1.71    book 89.87 / 14.7100 / 1.71  (-0.0 s.e.)


   0.80        94.23       12.8045 (0.0070)         1.49    book 94.23 / 12.8000 / 1.49  (-0.7 s.e.)


   0.95        99.62       10.6426 (0.0066)         1.24    book 99.62 / 10.6400 / 1.24  (-0.4 s.e.)


## 10. 封顶棘轮：例 34.10

每一期都是一个远期起始的平值看涨价差，因此每单位名义本金值
$c(1,1,\tau) - c(1,1+C,\tau)$，即一个常数，而不是标的水平的函数。
**价值随波动率上升而下降**：上限贴近远期，使这份合约成为波动率空头。
整个设计就在于此，它能以醒目的收益率卖出去，也正是因为这一点。

In [13]:
CAP = 0.08
print("三年期、每年重置的棘轮，每期封顶 %.0f%%" % (100 * CAP))
print("%12s %14s %20s" % ("forward vol", "per period", "three-year (3V)"))
for fv, wp, wt in ((0.15, 0.03800, 0.11401), (0.20, 0.03670, 0.11009), (0.25, 0.03561, 0.10682),
                   (0.30, 0.03463, 0.10388), (0.35, 0.03372, 0.10115)):
    V = bs(1.0, 1.0, 1.0, r, q, fv) - bs(1.0, 1.0 + CAP, 1.0, r, q, fv)
    print("%11.0f%% %14.5f %20.5f    book %.5f / %.5f" % (100 * fv, V, 3 * V, wp, wt))

print()
print("作为对照，一个不封顶的棘轮，如预期那样是波动率多头:")
for fv in (0.15, 0.25, 0.35):
    print("  前向波动率 %.0f%%   不封顶每期 %.5f" % (100 * fv, bs(1.0, 1.0, 1.0, r, q, fv)))

三年期、每年重置的棘轮，每期封顶 8%
 forward vol     per period      three-year (3V)
         15%        0.03800              0.11401    book 0.03800 / 0.11401
         20%        0.03670              0.11009    book 0.03670 / 0.11009
         25%        0.03561              0.10682    book 0.03561 / 0.10682
         30%        0.03463              0.10388    book 0.03463 / 0.10388
         35%        0.03372              0.10115    book 0.03372 / 0.10115

作为对照，一个不封顶的棘轮，如预期那样是波动率多头:
  前向波动率 15%   不封顶每期 0.08592
  前向波动率 25%   不封顶每期 0.12336
  前向波动率 35%   不封顶每期 0.16128
